In [2]:
def setup_environment() -> None:
    import os
    from dotenv import load_dotenv
    _ = load_dotenv()

    from huggingface_hub import login
    login(token=os.environ['HUGGINGFACEHUB_API_TOKEN'])
   
setup_environment()

In [18]:
import os
os.environ['HF_HOME'] = "/mnt/models-vol/cache_huggingface"
!echo $HF_HOME


/mnt/models-vol/cache_huggingface


In [1]:
cd /mnt/prj-vol

/__modal/volumes/vo-zgpcM6skMu8osC1WPr0Q7L


In [3]:
ls

tts-zipvoice-ft/


In [5]:
!git clone https://github.com/neuphonic/neutts.git

Cloning into 'neutts'...
remote: Enumerating objects: 324, done.
remote: Counting objects: 100% (205/205), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 324 (delta 135), reused 105 (delta 100), pack-reused 119 (from 1)
Receiving objects: 100% (324/324), 1.91 MiB | 17.78 MiB/s, done.
Resolving deltas: 100% (172/172), done.


In [8]:
cd neutts

/__modal/volumes/vo-zgpcM6skMu8osC1WPr0Q7L/neutts


In [11]:
!pip install -r requirements.txt

  Preparing metadata (setup.py) ... - done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 140.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 166.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 910.7/910.7 kB 173.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 134.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 122.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 134.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 142.2 MB/s eta 0:00:00
   ━━━

In [15]:
!apt install espeak-ng -y




The following additional packages will be installed:
  espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
The following NEW packages will be installed:
  espeak-ng espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 50 not upgraded.
Need to get 4829 kB of archives.
After this operation, 13.8 MB of additional disk space will be used.
Get:1 http://deb.debian.org/debian bookworm/main amd64 libpcaudio0 amd64 1.2-2 [8800 B]
Get:2 http://deb.debian.org/debian bookworm/main amd64 libsonic0 amd64 0.2.0-12 [11.0 kB]
Get:3 http://deb.debian.org/debian bookworm/main amd64 espeak-ng-data amd64 1.51+dfsg-10+deb12u2 [4256 kB]
Get:4 http://deb.debian.org/debian bookworm/main amd64 libespeak-ng1 amd64 1.51+dfsg-10+deb12u2 [200 kB]
Get:5 http://deb.debian.org/debian bookworm/main amd64 espeak-ng amd64 1.51+dfsg-10+deb12u2 [353 kB]
Fetched 4829 kB in 0s (22.7 MB/s)
debconf: delaying package configuration, since apt-utils is not installed

78Selecting pr

### Inference

In [29]:
%%writefile /mnt/prj-vol/neutts/examples/basic_example_1.py
import os
import soundfile as sf
from neutts import NeuTTS


def main(input_text, ref_audio_path, ref_text, backbone, output_path="output.wav"):
    if not ref_audio_path or not ref_text:
        print("No reference audio or text provided.")
        return None

    # Initialize NeuTTS with the desired model and codec
    tts = NeuTTS(
        backbone_repo=backbone,
        backbone_device="cuda",
        codec_repo="neuphonic/neucodec",
        codec_device="cuda",
    )

    # Check if ref_text is a path if it is read it if not just return string
    if ref_text and os.path.exists(ref_text):
        with open(ref_text, "r") as f:
            ref_text = f.read().strip()

    print("Encoding reference audio")
    ref_codes = tts.encode_reference(ref_audio_path)

    print(f"Generating audio for input text: {input_text}")
    wav = tts.infer(input_text, ref_codes, ref_text)

    print(f"Saving output to {output_path}")
    sf.write(output_path, wav, 24000)


if __name__ == "__main__":
    # get arguments from command line
    import argparse

    parser = argparse.ArgumentParser(description="NeuTTS Example")
    parser.add_argument(
        "--input_text", type=str, required=True, help="Input text to be converted to speech"
    )
    parser.add_argument(
        "--ref_audio", type=str, default="./samples/jo.wav", help="Path to reference audio file"
    )
    parser.add_argument(
        "--ref_text",
        type=str,
        default="./samples/jo.txt",
        help="Reference text corresponding to the reference audio",
    )
    parser.add_argument(
        "--output_path", type=str, default="output.wav", help="Path to save the output audio"
    )
    parser.add_argument(
        "--backbone",
        type=str,
        default="neuphonic/neutts-nano",
        help="Huggingface repo containing the backbone checkpoint",
    )
    args = parser.parse_args()
    main(
        input_text=args.input_text,
        ref_audio_path=args.ref_audio,
        ref_text=args.ref_text,
        backbone=args.backbone,
        output_path=args.output_path,
    )

Overwriting /mnt/prj-vol/neutts/examples/basic_example_1.py


In [31]:
!python -m examples.basic_example_1 \
  --input_text "My name is Andy. I'm 25 and I just moved to London. The underground is pretty confusing, but it gets me around in no time at all." \
  --ref_audio /mnt/data-vol/13-female-north-young-story-tuyet-trinh-vivi-11s.mp3 \
  --ref_text "Hello, I'm delighted to assist you with our voice services. Choose a voice that resonates with you, and let's begin our creative audio journey together"

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu129 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
Loading phonemizer...
Loading backbone from: neuphonic/neutts-nano on cuda ...
Loading codec from: neuphonic/neucodec on cuda ...
Fetching 1 files: 100%|█████████████████████████| 1/1 [00:00<00:00, 1676.38it/s]
/usr/local/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
loaded PerthNet (Implicit) at step 250,000
Encoding reference audio
Generating audio for input text: My name is Andy. I'm 25 and I just moved to London. The underground is pretty confusing, but it gets me around in no time at all.
Saving output to output.wav


### ONNX Inference

In [50]:
%%writefile /mnt/prj-vol/neutts/examples/onnx_example.py
import os
import soundfile as sf
import torch
from neutts import NeuTTS


def main(input_text, ref_codes_path, ref_text, backbone, output_path="output.wav"):
    if not ref_codes_path or not ref_text:
        print("No reference audio or text provided.")
        return None

    # Initialize NeuTTS with the desired model and codec
    tts = NeuTTS(
        backbone_repo=backbone,
        backbone_device="cuda",
        codec_repo="neuphonic/neucodec-onnx-decoder",
        codec_device="cuda",
    )

    # Check if ref_text is a path if it is read it if not just return string
    if ref_text and os.path.exists(ref_text):
        with open(ref_text, "r") as f:
            ref_text = f.read().strip()

    if ref_codes_path and os.path.exists(ref_codes_path):
        ref_codes = torch.load(ref_codes_path)

    print(f"Generating audio for input text: {input_text}")
    wav = tts.infer(input_text, ref_codes, ref_text)

    print(f"Saving output to {output_path}")
    sf.write(output_path, wav, 24000)


if __name__ == "__main__":
    # get arguments from command line
    import argparse

    parser = argparse.ArgumentParser(description="NeuTTS Example")
    parser.add_argument(
        "--input_text", type=str, required=True, help="Input text to be converted to speech"
    )
    parser.add_argument(
        "--ref_codes",
        type=str,
        default="./samples/jo.pt",
        help="Path to pre-encoded reference audio",
    )
    parser.add_argument(
        "--ref_text",
        type=str,
        default="./samples/jo.txt",
        help="Reference text corresponding to the reference audio",
    )
    parser.add_argument(
        "--output_path", type=str, default="output.wav", help="Path to save the output audio"
    )
    parser.add_argument(
        "--backbone",
        type=str,
        default="neuphonic/neutts-nano",
        help="Huggingface repo containing the backbone checkpoint",
    )
    args = parser.parse_args()
    main(
        input_text=args.input_text,
        ref_codes_path=args.ref_codes,
        ref_text=args.ref_text,
        backbone=args.backbone,
        output_path=args.output_path,
    )

Overwriting /mnt/prj-vol/neutts/examples/onnx_example.py


In [51]:
!python -m examples.onnx_example \
  --input_text "My name is Andy. I'm 25 and I just moved to London. The underground is pretty confusing, but it gets me around in no time at all." \
  --ref_codes /mnt/data-vol/13-female-north-young-story-tuyet-trinh-vivi-11s_ref_codes.pt \
  --ref_text "Hello, I'm delighted to assist you with our voice services. Choose a voice that resonates with you, and let's begin our creative audio journey together" \
  --backbone neuphonic/neutts-nano-q4-gguf

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu129 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
Loading phonemizer...
Loading backbone from: neuphonic/neutts-nano-q4-gguf on cuda ...
Loading codec from: neuphonic/neucodec-onnx-decoder on cuda ...
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/__modal/volumes/vo-zgpcM6skMu8osC1WPr0Q7L/neutts/examples/onnx_example.py", line 65, in <module>
    main(
  File "/__modal/volumes/vo-zgpcM6skMu8osC1WPr0Q7L/neutts/examples/onnx_example.py", line 13, in main
    tts = NeuTTS(
          ^^^^^^^
  File "/__modal/volumes/vo-zgpcM6skMu8osC1WPr0Q7L/neutts/neutts/neutts.py", line 108, in __init__
    self._load_codec(codec_repo, codec_device)
  File "/__modal/volumes/vo-zgpcM6skMu8osC1WPr0Q7L/neutts/neutts/neutts.py", line 196, in _load_codec
    raise ValueError(

In [35]:
!apt install -y portaudio19-dev python3-dev




The following additional packages will be installed:
  javascript-common libasound2-dev libexpat1-dev libjack-jackd2-dev libjs-jquery libjs-sphinxdoc
  libjs-underscore libportaudio2 libportaudiocpp0 libpython3-dev libpython3.11 libpython3.11-dev
  python3-distutils python3-lib2to3 python3.11-dev zlib1g-dev
Suggested packages:
  apache2 | lighttpd | httpd libasound2-doc portaudio19-doc
The following NEW packages will be installed:
  javascript-common libasound2-dev libexpat1-dev libjack-jackd2-dev libjs-jquery libjs-sphinxdoc
  libjs-underscore libportaudio2 libportaudiocpp0 libpython3-dev libpython3.11 libpython3.11-dev
  portaudio19-dev python3-dev python3-distutils python3-lib2to3 python3.11-dev zlib1g-dev
0 upgraded, 18 newly installed, 0 to remove and 50 not upgraded.
Need to get 9596 kB of archives.
After this operation, 41.4 MB of additional disk space will be used.
Get:1 http://deb.debian.org/debian bookworm/main amd64 javascript-common all 11+nmu1 [6260 B]
Get:2 http://deb.

In [36]:
!pip install pyaudio

  Installing build dependencies ... - \ | / done
  Getting requirements to build wheel ... - done
  Preparing metadata (pyproject.toml) ... - done
  Created wheel for pyaudio: filename=pyaudio-0.2.14-cp312-cp312-linux_x86_64.whl size=27786 sha256=3bf6028cb8b95000052af93369a611c868d3960051a8d232f30b82c19ce5a141
  Stored in directory: /tmp/pip-ephem-wheel-cache-9dxzbzrp/wheels/68/c7/33/c6a6b210cb5819ec5c219928c794a447742a7d86d21c0b92e6
Successfully built pyaudio

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [40]:
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 138.1 MB/s eta 0:00:00
  Installing build dependencies ... - \ | / done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... - \ done
  Preparing metadata (pyproject.toml) ... - done
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl size=4251282 sha256=6068cb77eb06b3f028e62e13f73bbe8b9abe4d5b398da94f1de1614920b51ce4
  Stored in directory: /tmp/pip-ephem-wheel-cache-uocu92bx/wheels/90/82/ab/8784ee3fb99ddb07fd36a679ddbe63122cc07718f6c1eb3be8
Successfully built llama-cpp-python

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


#### Streaming Inference

In [42]:
!pip install onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 38.4 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [48]:
!python -m examples.basic_streaming_example \
  --input_text "My name is Andy. I'm 25 and I just moved to London. The underground is pretty confusing, but it gets me around in no time at all." \
  --ref_codes /mnt/data-vol/13-female-north-young-story-tuyet-trinh-vivi-11s_ref_codes.pt \
  --ref_text "Hello, I'm delighted to assist you with our voice services. Choose a voice that resonates with you, and let's begin our creative audio journey together"

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu129 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
Loading phonemizer...
Loading backbone from: neuphonic/neutts-nano-q8-gguf on cpu ...
Loading codec from: neuphonic/neucodec-onnx-decoder on cpu ...
loaded PerthNet (Implicit) at step 250,000
Generating audio for input text: My name is Andy. I'm 25 and I just moved to London. The underground is pretty confusing, but it gets me around in no time at all.
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5180:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5180:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5180:(_snd_config_evaluate) f


#### Pre-encode a reference

File này dùng để mã hóa (encode) một đoạn audio tham chiếu thành các mã rời rạc (codes) bằng mô hình NeuCodec, rồi lưu các codes đó ra file .pt.
Nó không tạo ra âm thanh, mà chỉ chuyển audio → biểu diễn số học trung gian.

Nói ngắn gọn:
👉 Audio người nói → NeuCodec → tensor codes → lưu lại để dùng về sau

In [47]:
!python -m examples.encode_reference \
  --ref_audio /mnt/data-vol/13-female-north-young-story-tuyet-trinh-vivi-11s.mp3 \
  --output_path /mnt/data-vol/13-female-north-young-story-tuyet-trinh-vivi-11s_ref_codes.pt

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu129 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
Encoding reference audio
Fetching 1 files: 100%|█████████████████████████| 1/1 [00:00<00:00, 5398.07it/s]
/usr/local/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
